# 41 · Chronological inference parity

Pause model tuning and test the saved working recipe at its inference boundary.
Feed one completed epoch summary at a time. Keep at most thirty epochs of history.
Compare probabilities and predictions with offline validation on all five frozen development folds.

This experiment does not train, change checkpoints, or read final-test observations.
It uses the LR0.001 motion recipe, not the lower-rate candidate.
The input is a prepared epoch summary, not a raw Apple Watch sensor event.
The target label is used only for audit alignment; it is never supplied to the streaming predictor.
Raw sensor aggregation, missing-data decisions, and collection latency require later end-to-end validation.

Read the [protocol](../docs/chronological-inference.md). Restart the kernel before running this notebook.


## Locate the checkout and switches

**What:** Resolve the checkout and declare the optional inference audit.

**Why:** A report-only run must not create a snapshot or run inference.


In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("chronological_inference_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/chronological-inference.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()

REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/chronological-inference"  # Select the PR branch before merge
RUN_SETUP = False
RUN_AUDIT = False

logger.info("Repository: %s", ROOT)
logger.info("Setup=%s chronological inference audit=%s", RUN_SETUP, RUN_AUDIT)


## Optional environment setup

**What:** Clone the working branch and install locked uv groups when needed.

**Why:** The notebook and saved checkpoints need the same Python dependencies.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")

## Verify the frozen reference

**What:** Validate all five saved folds, model-result hashes, and the declared audit scope.

**Why:** A parity test must use the intended reference and retain its backend.


In [ ]:
import pandas as pd

from sleep_classifier.chronological_replay import ChronologicalReplayStudy

study = ChronologicalReplayStudy(ROOT / "configs/chronological-inference.yaml")
plan = study.plan()
if (len(plan["folds"]) != 5 or plan["development_participants"] != 26
        or plan["reserved_test_participants"] != 3 or plan["device"] != "mps"):
    raise ValueError("The frozen development scope or explicit MPS backend changed")
settings = study.reference.model_settings
if (settings["learning_rate"] != 0.001 or settings["seed"] != 42
        or settings["context_epochs"] != 30 or settings["patch_size"] != 3
        or settings["transformer"]["d_model"] != 16
        or settings["transformer"]["dropout"] != 0.2):
    raise ValueError("Keep the working motion recipe fixed")
logger.info("Reference=%s output=%s tolerance=%s", study.reference.output_dir,
            study.output_dir, study.settings["absolute_tolerance"])
display(pd.DataFrame([{"fold": name, **hashes} for name, hashes in plan["sources"].items()])
        .set_index("fold"))


## Run chronological inference only after opt-in

**What:** Replay prepared validation summaries with selected checkpoints and fixed preprocessing.

**Why:** This checks streaming parity without another model-training experiment.


In [ ]:
if RUN_AUDIT:
    if (study.output_dir / "report.json").is_file():
        study.read_report()
        logger.info("Verified cached replay snapshot; no inference repeated")
    else:
        audit_config = study.config_path
        logger.info("Audit config=%s output=%s reference=%s", audit_config,
                    study.output_dir, study.reference.output_dir)
        study.validate_run_start()
        %cd "{ROOT}"
        !uv run --locked --group gru --group notebook python -u \
            -m sleep_classifier.chronological_replay --config "{audit_config}" --run
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Replay failed; preserve its partial evidence before retrying")
else:
    logger.info("Inference audit skipped; inspect a saved report below when available")


## Inspect numerical parity and history resets

**What:** Load verified replay evidence and show discrepancies for each fold.

**Why:** Pass conditions test implementation equivalence; they do not measure a new model gain.


In [ ]:
report = study.read_report()
logger.info("Replay status=%s", report["status"])
if report["folds"]:
    rows = [
        {"fold": row["fold"], "status": row["status"], "epochs": row["epochs"],
         "stream_offline_max_abs": row["stream_vs_offline"]["max_abs_probability_difference"],
         "stream_offline_class_disagreements": row["stream_vs_offline"]["prediction_disagreements"],
         "offline_saved_max_abs": row["offline_vs_saved"]["max_abs_probability_difference"],
         "offline_saved_class_disagreements": row["offline_vs_saved"]["prediction_disagreements"],
         "history_matches": row["history_lengths_match"],
         "history_starts": row["history_start_epochs"],
         "full_context_epochs": row["full_context_epochs"],
         "weights_unchanged": row["weights_unchanged"],
         "gradients_created": row["gradients_created"]}
        for row in report["folds"]
    ]
    display(pd.DataFrame(rows).set_index("fold"))
    display(pd.DataFrame([{"fold": row["fold"], **row["reset_counts"]}
                          for row in report["folds"]]).fillna(0).set_index("fold"))
    logger.info("Scope: %s", report["scope"])
else:
    logger.info("No completed snapshot yet; enable RUN_AUDIT when ready")


## Visualize agreement and observed timing

**What:** Plot probability differences and per-epoch replay durations when available.

**Why:** Separate numerical parity from descriptive Python timing and deployment claims.


In [ ]:
import matplotlib.pyplot as plt

if report["folds"]:
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    names = [row["fold"] for row in report["folds"]]
    axes[0].bar(names, [row["stream_vs_offline"]["max_abs_probability_difference"]
                       for row in report["folds"]])
    axes[0].axhline(study.settings["absolute_tolerance"], color="red", linestyle="--",
                   label="Predeclared tolerance")
    axes[0].set_ylabel("Maximum absolute probability difference")
    axes[0].legend()
    axes[1].bar(names, [1000 * row["observed_p95_epoch_seconds"] for row in report["folds"]])
    axes[1].set_ylabel("Observed p95 epoch duration (ms)")
    axes[1].set_title("Python replay; not a deployment benchmark")
    figure.tight_layout()
    display(figure)
    plt.close(figure)
else:
    logger.info("Plots pending until the inference audit finishes")


## Decide what the audit establishes

A pass requires absolute probability differences at most 0.000002 and zero predicted-class disagreements.
It also requires unchanged weights, no gradients, and the same participant-local history lengths as offline inference.
The test compares offline inference with saved predictions to detect a stale checkpoint or changed input population.

A pass supports this prepared-feature inference interface. It does not prove that raw sensor ingestion is causal or timely.
A failure is evidence to inspect, not a reason to relax the tolerance automatically.
Keep failed snapshots and identify the mismatched targets before changing implementation or thresholds.
Do not rank models using this test or inspect the final-test participants.

After parity, consolidate the model card and inference contract. Resume modeling only for a new, specific hypothesis.


## Export the inspection report

**What:** Execute report-only cells in a fresh kernel and save HTML.

**Why:** A repeatable inspection report must not run inference or recursively export itself.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/41_chronological_inference.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Chronological inference report export failed")
